# Практическая работа № 3. Сборка пайплайна предобработки

**Уровень:** B (данные заданы)
**Цель:** собрать пайплайн предобработки, который **не допускает утечки** информации из тестовой выборки в обучающую.

**Данные:** `data/housing.csv`. Проблемы данных, найденные в ПР-2, здесь устраняются.

In [1]:
import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

df = pd.read_csv("../data/housing.csv")
print("Исходный размер:", df.shape)

Исходный размер: (1010, 11)


## 1. Правила очистки по здравому смыслу (до разбиения)
Дубликаты, площадь > 400 м² и невозможные годы постройки (см. ПР-2) убираются правилами, которые **не считают статистику по данным** (никаких средних, медиан). Поэтому применять их до разбиения безопасно: утечки нет. Невозможные значения заменяются на пропуск — их заполнит пайплайн уже по правилам, выученным только на train.

In [2]:
df = df.drop_duplicates().copy()
df.loc[df["area_m2"] > 400, "area_m2"] = np.nan
df.loc[~df["year_built"].between(1900, 2026), "year_built"] = np.nan
print("Размер после удаления дубликатов:", df.shape)
print("Пропусков в area_m2 теперь:", df["area_m2"].isnull().sum(), "| в year_built:", df["year_built"].isnull().sum())

Размер после удаления дубликатов: (1000, 11)
Пропусков в area_m2 теперь: 36 | в year_built: 83


## 2. Разбиение train / test
Разбиение выполняется **до** любых операций, считающих статистику (медиана, среднее, стандартное отклонение, список категорий).

In [3]:
X = df.drop(columns="price_kusd")
y = df["price_kusd"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("train:", X_train.shape, "| test:", X_test.shape)

train: (800, 10) | test: (200, 10)


## 3. Стратегия обработки для каждой группы признаков

| Признаки | Тип | Пропуски | Стратегия | Обоснование |
|---|---|---|---|---|
| `area_m2`, `living_area_m2`, `area_sqft` | числовые непрерывные | 3% (area_m2), остальные нет | **медиана** → StandardScaler | распределения скошены вправо; среднее смещается хвостом, медиана устойчива к выбросам |
| `dist_center_km` | числовой непрерывный | 5% | **медиана** → StandardScaler | одна и та же причина: устойчивость к выбросам |
| `rooms`, `floor` | числовые дискретные | `floor` 3% | **медиана** → StandardScaler | медиана остаётся «осмысленным» значением (целым этажом); среднее дало бы дробный этаж |
| `year_built` | числовой | 8% (+3 невозможных) | **медиана** → StandardScaler | год распределён равномерно, медиана ≈ середина диапазона; среднее не лучше, но на невозможные значения медиана не реагирует |
| `district`, `condition` | категориальные | 6% и 10% | **отдельная категория «неизвестно»** → OneHotEncoder | среднее для категорий не определено; 0 как «категория» — ошибка. Пропуск сам по себе может быть информативным, поэтому модель должна его «видеть» |
| `balcony` | категориальный бинарный | нет | OneHotEncoder | признак номинальный; пропусков нет, но стратегия на случай их появления в новых данных задана |

Масштабирование нужно линейным моделям и градиентному спуску (ПР-4): без него признак `area_sqft` (сотни) «перекрикивал» бы `rooms` (единицы). Для деревьев (ПР-7) оно не вредит.

In [4]:
num_cols = ["area_m2", "living_area_m2", "area_sqft", "rooms", "floor", "year_built", "dist_center_km"]
cat_cols = ["district", "condition", "balcony"]

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="неизвестно")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols),
])
preprocessor

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['area_m2', 'living_area_m2', 'area_sqft',
                                  'rooms', 'floor', 'year_built',
                                  'dist_center_km']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value='неизвестно',
                                                                strategy='constant')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['district', 'condition', 'balcony'])])

## 4. Обучение пайплайна только на train, применение к test
`fit_transform` — только к train; к test применяется **только `transform`** (без повторного обучения).

In [5]:
X_train_prep = preprocessor.fit_transform(X_train)   # fit — только на train
X_test_prep = preprocessor.transform(X_test)         # на test — только transform

print("Форма train после предобработки:", X_train_prep.shape)
print("Форма test после предобработки :", X_test_prep.shape)
print("Число столбцов совпадает:", X_train_prep.shape[1] == X_test_prep.shape[1])
print("Остались пропуски? train:", np.isnan(X_train_prep.toarray() if hasattr(X_train_prep, "toarray") else X_train_prep).any(),
      "| test:", np.isnan(X_test_prep.toarray() if hasattr(X_test_prep, "toarray") else X_test_prep).any())

Форма train после предобработки: (800, 18)
Форма test после предобработки : (200, 18)
Число столбцов совпадает: True
Остались пропуски? train: False | test: False


In [6]:
feature_names = preprocessor.get_feature_names_out()
print(len(feature_names), "признаков на выходе:")
print(list(feature_names))

18 признаков на выходе:
['num__area_m2', 'num__living_area_m2', 'num__area_sqft', 'num__rooms', 'num__floor', 'num__year_built', 'num__dist_center_km', 'cat__district_Восток', 'cat__district_Север', 'cat__district_Центр', 'cat__district_Юг', 'cat__district_неизвестно', 'cat__condition_неизвестно', 'cat__condition_новый', 'cat__condition_требует ремонта', 'cat__condition_хороший', 'cat__balcony_да', 'cat__balcony_нет']


## 5. Проверка отсутствия утечки
Что должно выполняться, если утечки нет:
1. Параметры (медианы, средние, стандартные отклонения) вычислены **по train**, а не по всем данным.
2. После преобразования среднее у train равно 0, а у test — **не равно** 0 (оно тоже посчитано с параметрами train). Если бы у test тоже было точно 0, это значило бы, что scaler переобучился на test.
3. Число столбцов у train и test одинаково, даже если в test встречаются новые категории.

In [7]:
scaler = preprocessor.named_transformers_["num"].named_steps["scaler"]
imputer = preprocessor.named_transformers_["num"].named_steps["imputer"]

# Проверка 1: статистики выучены на train
print("Медиана area_m2: train =", imputer.statistics_[0], "| вся выборка =", X["area_m2"].median())
print("Среднее (после заполнения) area_m2 у scaler:", round(scaler.mean_[0], 2), "(посчитано по train)")

# Проверка 2: среднее масштабированных признаков
dense_train = X_train_prep.toarray() if hasattr(X_train_prep, "toarray") else X_train_prep
dense_test = X_test_prep.toarray() if hasattr(X_test_prep, "toarray") else X_test_prep
n_num = len(num_cols)
print("\nСреднее числовых признаков, train:", dense_train[:, :n_num].mean(axis=0).round(3))
print("Среднее числовых признаков, test :", dense_test[:, :n_num].mean(axis=0).round(3))

Медиана area_m2: train = 60.8 | вся выборка = 61.05
Среднее (после заполнения) area_m2 у scaler: 64.09 (посчитано по train)

Среднее числовых признаков, train: [-0. -0. -0. -0. -0. -0. -0.]
Среднее числовых признаков, test : [ 0.089  0.102  0.084  0.078 -0.081  0.057 -0.064]


**Вывод по проверкам:** у train средние равны 0 (с точностью до округления), а у test они небольшие, но отличны от нуля — значит, test преобразован с параметрами, выученными на train, то есть утечки нет. Различие между медианой по train и по всей выборке тоже видно в первой проверке: если бы мы считали медиану по всем данным до разбиения, test «подсмотрел» бы в train.

In [8]:
# Проверка 3: новая, ни разу не виденная категория
new_row = X_test.iloc[[0]].copy()
new_row["district"] = "Запад"          # такого района нет в train
new_row["condition"] = np.nan           # пропуск в категориальном признаке
res = preprocessor.transform(new_row)
res = res.toarray() if hasattr(res, "toarray") else res
print("Форма результата для строки с новым районом:", res.shape, "(столбцов столько же, сколько у train:", X_train_prep.shape[1], ")")

Форма результата для строки с новым районом: (1, 18) (столбцов столько же, сколько у train: 18 )


## 6. Итоговый вывод

- Данные разбиты на train и test **до** вычисления статистик; правила очистки без статистики (дубликаты, невозможные значения) применены до разбиения без риска утечки.
- Все шаги объединены в `ColumnTransformer` с вложенными `Pipeline`: числовые признаки — медиана + стандартизация; категориальные — категория «неизвестно» + one-hot (`handle_unknown="ignore"`).
- Пайплайн обучен только на train и применён (`transform`) к test; размерности совпадают: train и test имеют одинаковое число признаков после преобразования, пропусков не осталось, новые категории не ломают пайплайн.
- Этот же пайплайн (код вынесен в `src/preprocessing.py`) используется в ПР-4 и ПР-7.